# Deep Learning Regularization & Optimization

Techniques to **prevent overfitting** and **train faster** in neural networks:

- **Dropout**: Randomly deactivate neurons during training
- **Batch Normalization**: Normalize layer inputs for stability
- **L1/L2 Regularization** (Weight Decay)
- **Early Stopping**: Stop when validation loss stops improving
- **Learning Rate Scheduling**: Reduce LR over time
- **Data Augmentation**: Artificially expand training data

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# Load Fashion MNIST
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Use subset for faster training
X_train_sub = X_train[:5000]
y_train_sub = y_train_cat[:5000]
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'Train: {X_train_sub.shape} | Test: {X_test.shape}')

## 1. Baseline Model (No Regularization)

In [ ]:
def build_baseline():
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(512, activation='relu'),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

baseline = build_baseline()
h_baseline = baseline.fit(X_train_sub, y_train_sub, epochs=50, batch_size=32, 
                           validation_data=(X_test, y_test_cat), verbose=0)
print(f'Baseline — Train: {h_baseline.history["accuracy"][-1]:.4f} | Val: {h_baseline.history["val_accuracy"][-1]:.4f}')

## 2. Dropout Regularization

Randomly sets a fraction of neurons to 0 during training → forces the network to learn redundant representations.

In [ ]:
def build_dropout(rate=0.3):
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(256, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(128, activation='relu'),
        layers.Dropout(rate),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

dropout_model = build_dropout(0.3)
h_dropout = dropout_model.fit(X_train_sub, y_train_sub, epochs=50, batch_size=32,
                               validation_data=(X_test, y_test_cat), verbose=0)
print(f'Dropout — Train: {h_dropout.history["accuracy"][-1]:.4f} | Val: {h_dropout.history["val_accuracy"][-1]:.4f}')

## 3. Batch Normalization

Normalizes the input of each layer → stabilizes training, allows higher learning rates.

In [ ]:
def build_batchnorm():
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(512), layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dense(256), layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dense(128), layers.BatchNormalization(), layers.Activation('relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

bn_model = build_batchnorm()
h_bn = bn_model.fit(X_train_sub, y_train_sub, epochs=50, batch_size=32,
                     validation_data=(X_test, y_test_cat), verbose=0)
print(f'BatchNorm — Train: {h_bn.history["accuracy"][-1]:.4f} | Val: {h_bn.history["val_accuracy"][-1]:.4f}')

## 4. L2 Regularization (Weight Decay)

In [ ]:
def build_l2(l2_val=0.001):
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(l2_val)),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_val)),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_val)),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

l2_model = build_l2(0.001)
h_l2 = l2_model.fit(X_train_sub, y_train_sub, epochs=50, batch_size=32,
                     validation_data=(X_test, y_test_cat), verbose=0)
print(f'L2 Reg — Train: {h_l2.history["accuracy"][-1]:.4f} | Val: {h_l2.history["val_accuracy"][-1]:.4f}')

## 5. Early Stopping + LR Scheduling

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
lr_scheduler = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

es_model = build_baseline()
h_es = es_model.fit(X_train_sub, y_train_sub, epochs=100, batch_size=32,
                     validation_data=(X_test, y_test_cat), verbose=0,
                     callbacks=[early_stop, lr_scheduler])
actual_epochs = len(h_es.history['loss'])
print(f'Early Stopping — Stopped at epoch {actual_epochs}')
print(f'Train: {h_es.history["accuracy"][-1]:.4f} | Val: {h_es.history["val_accuracy"][-1]:.4f}')

## 6. Combined — Best Practices Model

In [ ]:
def build_best_practice():
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28, 1)),
        layers.Dense(512, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(256, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
        layers.Dense(128, kernel_regularizer=regularizers.l2(0.001)),
        layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

best_model = build_best_practice()
h_best = best_model.fit(X_train_sub, y_train_sub, epochs=100, batch_size=32,
                         validation_data=(X_test, y_test_cat), verbose=0,
                         callbacks=[early_stop, lr_scheduler])
print(f'Best Practice — Train: {h_best.history["accuracy"][-1]:.4f} | Val: {h_best.history["val_accuracy"][-1]:.4f}')

## 7. Comparison

In [ ]:
histories = {
    'Baseline': h_baseline, 'Dropout': h_dropout, 'BatchNorm': h_bn,
    'L2 Reg': h_l2, 'Early Stop': h_es, 'Combined': h_best
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, h in histories.items():
    axes[0].plot(h.history['val_accuracy'], label=name, alpha=0.8)
    axes[1].plot(h.history['val_loss'], label=name, alpha=0.8)

axes[0].set_title('Validation Accuracy', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Loss', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Regularization Techniques — Comparison', fontsize=14)
plt.tight_layout()
plt.show()

# Summary table
import pandas as pd
summary = pd.DataFrame({
    'Method': list(histories.keys()),
    'Train Acc': [h.history['accuracy'][-1] for h in histories.values()],
    'Val Acc': [h.history['val_accuracy'][-1] for h in histories.values()],
    'Overfit Gap': [h.history['accuracy'][-1] - h.history['val_accuracy'][-1] for h in histories.values()]
})
print(summary.to_string(index=False))

## Cheat Sheet

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **Dropout** | Randomly disables neurons | Overfitting in Dense layers |
| **BatchNorm** | Normalizes layer inputs | Always (stabilizes training) |
| **L2 Reg** | Penalizes large weights | Overfitting, small datasets |
| **Early Stopping** | Stops training at best val loss | Always (free regularization) |
| **LR Schedule** | Reduces learning rate | Training plateaus |
| **Data Augmentation** | Augments training images | Image/CV tasks |